# MMA3001 | Formula 1 Next-Lap-Time Prediction
### Comparative Evaluation of Machine Learning Models for Formula 1 Lap Time Prediction and Race Strategy Analysis

**Engineering task:** At the end of lap *n*, predict the duration of **lap n+1**, conditional on the target being a normal consecutive racing lap in the same stint. This is a retrospective **clean-lap** experiment, **not** a prediction of safety cars, pit stops, DNFs or real-time reliability.

**Primary output:** `PredictedNextLapTime_s` (seconds); **regression target:** `DeltaLapTime_s = NextLapTime_s - PreviousLapTime_s`. Compare persistence, multiple linear regression, decision tree, and random forest. Evaluate a pace/context feature set against pace/context + **previous-lap** telemetry, with race/season-separated validation.

**References:** Uploaded *MMA3001 Codex Development Plan* (11 Oct 2026) and *MMA3001 2026 Project Brief*. Data source: [FastF1](https://docs.fastf1.dev/). All dataset statistics and accuracy figures are computed from **your real exports** at runtime; none are fabricated.

**Execution guide:** Run from top to bottom. Set `DATA_DIR` and split years in Section 2. The notebook does **not** download data unless you explicitly set `RUN_SINGLE_RACE_DOWNLOAD=True`. Run `RUN_FINAL_TEST=False` while developing; switch to `True` only after finishing model selection. Leave all collected FastF1 raw data/cache out of a public Git repository until licence conditions are checked.

> **Current dataset:** Use `f1_export/2024_bahrain_next_lap_ml.csv`. A single 2024 race supports ingestion/QA/EDA only; full model comparison needs additional years for chronological separation. The single-race file must contain actual lap-pair rows.


## 1 | Imports, environment and reproducibility

In [ ]:
from __future__ import annotations

# Standard library
import json
import os
import platform
import random
import sys
import time
import warnings
from pathlib import Path

# Third-party: numerical analysis and visualisation
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Third-party: supervised learning
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
print({"Python": sys.version.split()[0], "OS": platform.platform(),
       "pandas": pd.__version__, "scikit-learn": sklearn.__version__})
try:
    import torch
    print("PyTorch CUDA available:", torch.cuda.is_available(),
          "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
except ImportError:
    print("PyTorch not installed; not needed for scikit-learn models.")

## 2 | Project configuration and existing export location

The batch exporter writes data under `f1_multi_export/drivers_HAM_VER/`, including `combined_next_lap_ml.csv`, `export_summary.csv` and `batch_manifest.json`. The notebook **does not download or delete data automatically**. First inspect the real export status; an existing CSV with zero records is not a successful export. Use `RUN_SINGLE_RACE_DOWNLOAD=False` unless you intentionally opt into the separate fallback. Keep the final chronological 2022–23 / 2024 / 2025 split unchanged until sufficient actual race data exists.

In [ ]:
# Locate the repository reliably from either the project root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "mma3001_f1").is_dir() and (ROOT.parent / "mma3001_f1").is_dir():
    ROOT = ROOT.parent

# Single-race export: 2024_bahrain_next_lap_ml.csv lives under f1_export/.
# Change this path only when your exporter writes to a different location.
DATA_DIR = ROOT / "f1_export"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
MODEL_DIR = RESULTS_DIR / "models"
METRIC_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
for folder in [FIG_DIR, MODEL_DIR, METRIC_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_YEARS = (2022, 2023)
VAL_YEARS = (2024,)
TEST_YEARS = (2025,)
RUN_FINAL_TEST = False

# Leave disabled: FastF1 acquisition and existing exporter recovery should be intentional.
RUN_SINGLE_RACE_DOWNLOAD = False
DOWNLOAD_YEAR, DOWNLOAD_EVENT, DOWNLOAD_DRIVERS = 2024, "Bahrain", ("VER", "HAM")

REQUIRE_TELEMETRY = False
MAX_RF_TREES = 150
print("Project root:", ROOT)
print("Existing export folder:", DATA_DIR)
print("Export folder exists:", DATA_DIR.is_dir())


In [ ]:
# Export health check BEFORE attempting to load/training on lap-pair data.
# This inspects your existing batch output without changing, overwriting or deleting it.
summary_path = DATA_DIR / "export_summary.csv"
manifest_path = DATA_DIR / "batch_manifest.json"
combined_path = DATA_DIR / "combined_next_lap_ml.csv"
raw_path = DATA_DIR / "combined_all_laps.csv"

if summary_path.exists():
    export_summary = pd.read_csv(summary_path)
    print("Export status for each requested race:")
    display(export_summary)
    if "Status" in export_summary:
        print("Export statuses:", export_summary["Status"].value_counts(dropna=False).to_dict())
    if "Reason" in export_summary:
        for _, record in export_summary.iterrows():
            if str(record.get("Status", "")).lower() not in {"success", "cached", "complete"}:
                print(f"{record.get('Event', 'Unknown event')}: {record.get('Reason', 'Reason unavailable')}")
else:
    print("No export_summary.csv found at", summary_path)

if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as fh:
        batch_manifest = json.load(fh)
    print("Batch manifest available at:", manifest_path)
else:
    print("Batch manifest missing:", manifest_path)

for file in (combined_path, raw_path):
    if file.exists():
        data_probe = pd.read_csv(file)
        print(f"{file.name}: {len(data_probe):,} rows, {len(data_probe.columns)} columns")
    else:
        print(f"{file.name}: not found")

print("\nIf exports have failed, investigate the *single-race* extractor from the project root:")
print("python -m dataset.fastf1_export_dataset --year 2024 --event Bahrain "
      "--drivers VER HAM --output .\\f1_bahrain_test")
print("This command is for the VS Code PowerShell terminal, NOT automatic execution here.")


### Recovery after a failed batch export (PowerShell)

Your October pilot showed `failed` for Bahrain/Japan/Monaco and `MLRows=0`, so the **extractor must be debugged before ML training**. Open a VS Code PowerShell terminal in the project root and run:

```powershell
.\.venv\Scripts\Activate.ps1
python -m dataset.fastf1_export_dataset --year 2024 --event Bahrain --drivers VER HAM --output .\f1_bahrain_test
```

This uses a **new output folder**, preserving your prior failed batch and FastF1 cache. Check the traceback if it fails. Once you have a successful export with rows, set `DATA_DIR` to the folder containing its `*next_lap_ml*.csv`, rerun the notebook from configuration, and verify QA before any wider download. Do not use `--overwrite` or delete caches while investigating.

## 3 | Optional one-race FastF1 acquisition

This is a **fallback**, not a replacement for existing repository exporters (`fastf1_export_dataset.py`, `fastf1_export_multi_gp.py`). Your planning document says those scripts should be preserved; they are **not attached here**, so their actual behavior cannot be audited. Prefer their established `next_lap_ml` exports. The fallback below downloads **one** race only after explicit opt-in, and writes a separate, clearly named CSV. FastF1's network host may sometimes be blocked; that does not invalidate local-CSV modelling.

Optional installation on your own machine: `python -m pip install fastf1`. Basic fallback deliberately omits telemetry instead of pretending it is available. If weather timestamps cannot be verified to be prior to the source lap, temperature is left missing.

In [ ]:
def download_one_event(year: int, event: str, drivers: tuple[str, ...], output: Path) -> Path:
    """Download one race's basic clean consecutive-lap pairs (no telemetry).

    Parameters
    ----------
    year, event, drivers : int, str, tuple[str,...]
        FastF1 race identifier and optional driver abbreviation filter.
    output : Path
        New or overwritten opt-in fallback export path.
    """
    import fastf1  # optional dependency

    cache = ROOT / ".fastf1_cache"
    cache.mkdir(exist_ok=True)
    fastf1.Cache.enable_cache(str(cache))
    session = fastf1.get_session(year, event, "R")
    session.load(laps=True, telemetry=False, weather=False, messages=False)
    laps = session.laps.copy()
    if drivers:
        laps = laps[laps["Driver"].isin(drivers)].copy()
    laps = laps.sort_values(["Driver", "LapNumber"])
    records = []
    for driver, grp in laps.groupby("Driver"):
        arr = grp.sort_values("LapNumber").reset_index(drop=True)
        for i in range(len(arr)-1):
            a, z = arr.iloc[i], arr.iloc[i+1]
            if pd.isna(a.get("LapTime")) or pd.isna(z.get("LapTime")):
                continue
            if int(z["LapNumber"]) != int(a["LapNumber"]) + 1:
                continue
            if pd.isna(a.get("Stint")) or pd.isna(z.get("Stint")) or a["Stint"] != z["Stint"]:
                continue
            if pd.notna(a.get("PitInTime")) or pd.notna(z.get("PitOutTime")):
                continue
            if a.get("TrackStatus") != "1" or z.get("TrackStatus") != "1":
                continue
            if a.get("IsAccurate") != True or z.get("IsAccurate") != True:
                continue
            prev = a["LapTime"].total_seconds()
            nxt = z["LapTime"].total_seconds()
            records.append({"Year": year, "Event": event, "Session": "R", "Driver": driver,
                "PreviousLapNumber": int(a["LapNumber"]), "TargetLapNumber": int(z["LapNumber"]),
                "Stint": int(a["Stint"]), "PreviousLapTime_s": prev,
                "NextLapTime_s": nxt, "DeltaLapTime_s": nxt-prev,
                "TyreCompound": a.get("Compound"), "TyreAge_laps": a.get("TyreLife"),
                "TrackTemp_C": np.nan, "AvgSpeed_kmh": np.nan, "MaxSpeed_kmh": np.nan,
                "FullThrottle_pct": np.nan, "Braking_pct": np.nan})
    if not records:
        raise RuntimeError("No usable normal-lap pairs exported; check event/session coverage.")
    output.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(records).to_csv(output, index=False)
    print(f"Saved {len(records)} historical clean-lap pairs to {output}")
    return output

if RUN_SINGLE_RACE_DOWNLOAD:
    download_one_event(DOWNLOAD_YEAR, DOWNLOAD_EVENT, DOWNLOAD_DRIVERS,
                       DATA_DIR / f"fallback_{DOWNLOAD_YEAR}_{DOWNLOAD_EVENT}_next_lap_ml.csv")
else:
    print("Network acquisition disabled. Loading previously exported CSVs only.")

## 4 | Import verified real next-lap CSV data

Prefer the **combined** `combined_next_lap_ml.csv` in the selected batch output folder; do not concatenate it with individual per-race files, which would duplicate observations. If only per-race files exist, import those instead. Reject an entirely empty export with an actionable message and check `export_summary.csv` for failed session details. Do not fabricate F1 data or bypass the QA checks.

In [ ]:
REQUIRED_COLUMNS = ["Year", "Event", "Driver", "PreviousLapNumber", "TargetLapNumber",
                    "Stint", "PreviousLapTime_s", "NextLapTime_s"]
NUMERIC_FIELDS = ["Year", "PreviousLapNumber", "TargetLapNumber", "Stint", "PreviousLapTime_s",
                  "NextLapTime_s", "DeltaLapTime_s", "TyreAge_laps", "TrackTemp_C",
                  "AvgSpeed_kmh", "MaxSpeed_kmh", "FullThrottle_pct", "Braking_pct"]
OPTIONAL_COLUMNS = ["Session", "DeltaLapTime_s", "TyreCompound", "TyreAge_laps", "TrackTemp_C",
                    "AvgSpeed_kmh", "MaxSpeed_kmh", "FullThrottle_pct", "Braking_pct"]
KEYS = ["Year", "Event", "Session", "Driver", "PreviousLapNumber", "TargetLapNumber"]


def find_exports(folder: Path) -> list[Path]:
    """Select the batch aggregate once, or nonempty per-race exports if absent.

    The combined CSV and source per-race CSVs contain overlapping observations;
    reading both creates duplicate keys and misleading counts.
    """
    if not folder.is_dir():
        return []
    aggregate = folder / "combined_next_lap_ml.csv"
    if aggregate.is_file() and not pd.read_csv(aggregate, nrows=1).empty:
        return [aggregate]
    candidates = sorted(p for p in folder.rglob("*.csv")
                        if "next_lap_ml" in p.name.lower() and not p.name.lower().startswith("combined_")
                        and "summary" not in p.name.lower())
    return [p for p in candidates if not pd.read_csv(p, nrows=1).empty]


def load_exports(folder: Path) -> pd.DataFrame:
    """Load valid historical CSV exports, with source file for provenance.

    Raises
    ------
    FileNotFoundError
        If no usable completed ML export exists in the selected directory.
    ValueError
        If a CSV is malformed or contains no usable observations.
    """
    paths = find_exports(folder)
    if not paths:
        details = ""
        status_file = folder / "export_summary.csv"
        if status_file.is_file():
            status = pd.read_csv(status_file)
            if {"Event", "Status"}.issubset(status.columns):
                details = " Export summary: " + "; ".join(
                    f"{row.Event}: {row.Status}" for row in status.itertuples(index=False))
        raise FileNotFoundError(
            f"No populated *next_lap_ml*.csv found under {folder.resolve()}."
            f"{details} Run one race exporter from the repository root, inspect its error/manifest, "
            "then rerun the notebook. Do not change QA to hide missing data."
        )
    frames = []
    for path in paths:
        frame = pd.read_csv(path, low_memory=False)
        absent = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if absent:
            raise ValueError(f"{path}: required columns missing: {absent}")
        if frame.empty:
            continue
        frame["SourceFile"] = str(path.resolve())
        frames.append(frame)
    if not frames:
        raise ValueError("Matching export files exist but have no observations.")
    result = pd.concat(frames, ignore_index=True)
    for col in OPTIONAL_COLUMNS:
        if col not in result.columns:
            result[col] = np.nan if col != "Session" else "R"
    result["Session"] = result["Session"].fillna("R").astype(str)
    for col in NUMERIC_FIELDS:
        if col in result:
            result[col] = pd.to_numeric(result[col], errors="coerce")
    for col in ("Event", "Driver", "TyreCompound"):
        result[col] = result[col].astype("string").str.strip()
    return result


raw = load_exports(DATA_DIR)
print(f"Loaded {len(raw):,} rows from {raw['SourceFile'].nunique()} CSV file(s)")
display(raw.head(5))


## 5 | Quality control, leakage safeguards and reproducible clean-lap dataset

In [ ]:
def audit_pairs(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reject malformed pairs; return (clean rows, exclusion count per reason).

    Only input columns on an explicit allowlist enter training later. The historical
    future lap is used as the regression target and for retrospective selection only.
    """
    df = data.copy()
    # Keep earliest representation for a duplicated exported pair; flag duplicates.
    duplicated = df.duplicated(KEYS, keep="first")
    reasons = pd.DataFrame(index=df.index)
    reasons["duplicate_key"] = duplicated
    reasons["missing_identifiers"] = df[KEYS].isna().any(axis=1)
    reasons["non_consecutive"] = (df["TargetLapNumber"] != df["PreviousLapNumber"] + 1)
    reasons["invalid_time"] = (~df["PreviousLapTime_s"].between(30, 300) |
                               ~df["NextLapTime_s"].between(30, 300))
    reasons["invalid_stint"] = ~df["Stint"].between(1, 100)
    reasons["invalid_tyre_age"] = df["TyreAge_laps"].notna() & ~df["TyreAge_laps"].between(0, 100)
    reasons["invalid_speed"] = ((df["AvgSpeed_kmh"].notna() & ~df["AvgSpeed_kmh"].between(0, 400)) |
                                 (df["MaxSpeed_kmh"].notna() & ~df["MaxSpeed_kmh"].between(0, 400)))
    reasons["invalid_percentages"] = ((df["FullThrottle_pct"].notna() & ~df["FullThrottle_pct"].between(0, 100)) |
                                      (df["Braking_pct"].notna() & ~df["Braking_pct"].between(0, 100)))
    reasons["invalid_temperature"] = df["TrackTemp_C"].notna() & ~df["TrackTemp_C"].between(-10, 80)
    if "DeltaLapTime_s" in df:
        mismatch = df["DeltaLapTime_s"].notna() & ~np.isclose(
            df["DeltaLapTime_s"].fillna(0),
            (df["NextLapTime_s"]-df["PreviousLapTime_s"]).fillna(0), atol=1e-5)
        reasons["wrong_delta_identity"] = mismatch
    # Reject any row that violates at least one rule.
    reasons = reasons.fillna(True).astype(bool)
    clean = df.loc[~reasons.any(axis=1)].copy()
    clean["DeltaLapTime_s"] = clean["NextLapTime_s"] - clean["PreviousLapTime_s"]
    clean = clean.sort_values(KEYS).reset_index(drop=True)
    report = pd.DataFrame({"reason": list(reasons.columns), "excluded_rows": reasons.sum().astype(int).values})
    return clean, report



clean, exclusions = audit_pairs(raw)
print(f"Rows: original={len(raw):,}, retained={len(clean):,}, excluded={len(raw)-len(clean):,}")
display(exclusions)
if clean.empty:
    raise ValueError("No records passed QA. Inspect export format and exclusion report.")
# Missing values are imputed inside train-only fitted Pipelines, NEVER using held-out statistics.
display(clean[NUMERIC_FIELDS].isna().mean().mul(100).round(1).rename("missing_%").to_frame())
display(clean.groupby(["Year", "Event"], dropna=False).size().rename("clean_pairs").reset_index())
clean.to_csv(RESULTS_DIR / "next_lap_dataset.csv", index=False)
exclusions.to_csv(METRIC_DIR / "data_quality_exclusions.csv", index=False)

**Data lineage and limitation.** This notebook validates **exported pair identifiers**, continuity, units and target arithmetic; it cannot independently prove that the upstream exporter correctly identified pit-stop transitions, weather timestamp ordering, telemetry sampling or true target-lap track status. Those checks must be performed against original FastF1 lap records and exporter code before making deployment claims. Historially filtering for a clean target lap also makes this **conditional, retrospective** evaluation rather than an all-laps forecast.

## 6 | Verification tests (deterministic, not empirical model scores)

In [ ]:
def run_offline_checks() -> None:
    """Small hand-computable correctness checks for pairing and arithmetic."""
    toy = pd.DataFrame([
        {"Year": 2023,"Event":"TestGP","Session":"R","Driver":"AAA","PreviousLapNumber": 4,
         "TargetLapNumber": 5,"Stint": 1,"PreviousLapTime_s": 90.,"NextLapTime_s": 91.,
         "DeltaLapTime_s": 1.,"TyreAge_laps": 4},
        {"Year": 2023,"Event":"TestGP","Session":"R","Driver":"AAA","PreviousLapNumber": 5,
         "TargetLapNumber": 8,"Stint": 1,"PreviousLapTime_s": 91.,"NextLapTime_s": 92.,
         "DeltaLapTime_s": 1.,"TyreAge_laps": 5},
    ])
    for col in OPTIONAL_COLUMNS:
        if col not in toy:
            toy[col] = np.nan
    out, report = audit_pairs(toy)
    assert len(out) == 1
    assert out.iloc[0]["TargetLapNumber"] == 5
    assert report.set_index("reason").loc["non_consecutive", "excluded_rows"] == 1
    assert out.iloc[0]["DeltaLapTime_s"] == 1.
    # Persistence predicts delta zero, hence next lap equals previous lap.
    assert out.iloc[0]["PreviousLapTime_s"] + 0 == 90.
    # Check known exact MAE and RMSE values.
    assert mean_absolute_error([1, 3], [2, 2]) == 1.
    assert np.isclose(np.sqrt(mean_squared_error([1, 3], [2, 2])), 1.)
    print("Offline verification checks: PASS (pair continuity, delta identity, persistence, MAE, RMSE).")

run_offline_checks()

## 7 | Exploratory data analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(clean["DeltaLapTime_s"], bins=55)
axes[0].set(xlabel="Next − previous lap time (s)", ylabel="Count", title="Next-lap pace changes")
axes[1].scatter(clean["PreviousLapNumber"], clean["NextLapTime_s"], s=8, alpha=.3)
axes[1].set(xlabel="Completed lap number", ylabel="Next lap time (s)", title="Lap number vs next-lap time")
axes[2].scatter(clean["TyreAge_laps"], clean["DeltaLapTime_s"], s=8, alpha=.3)
axes[2].set(xlabel="Completed-lap tyre age (laps)", ylabel="Next lap − prior lap (s)", title="Tyre age vs pace change")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_overview.png", dpi=160)
plt.show()

## 8 | Lock race/season-separated train / validation / test groups

In [ ]:
def split_by_season(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fixed chronological seasons, with no shared event IDs across splits."""
    train = df.loc[df["Year"].isin(TRAIN_YEARS)].copy()
    val = df.loc[df["Year"].isin(VAL_YEARS)].copy()
    test = df.loc[df["Year"].isin(TEST_YEARS)].copy()
    groups = [set(zip(t["Year"], t["Event"], t["Session"])) for t in (train,val,test)]
    assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2]), "Event leakage"
    if train.empty or val.empty:
        raise ValueError("Insufficient seasons for training/validation. Update TRAIN_YEARS and VAL_YEARS "
            "to disjoint, chronological seasons you have actually downloaded; do not row-random-split.")
    if max(TRAIN_YEARS) >= min(VAL_YEARS) or max(VAL_YEARS) >= min(TEST_YEARS):
        raise ValueError("Season order must be TRAIN < VALIDATION < TEST.")
    return train, val, test

train_df, val_df, test_df = split_by_season(clean)
manifest = {"train": {"years": list(TRAIN_YEARS), "rows": len(train_df)},
            "validation": {"years": list(VAL_YEARS), "rows": len(val_df)},
            "test": {"years": list(TEST_YEARS), "rows": len(test_df)}}
(METRIC_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
if test_df.empty:
    print("NOTE: holdout data not present. Validation works; no final generalisation claim possible.")

## 9 | Model feature sets and preprocessing

**A**: persistence baseline: `next = previous`.

**B**: observed previous lap time + tyre compound/age + track temperature + completed lap index + circuit/event identity.

**C**: B plus **previous-lap** average/max speed, full throttle and braking percentages.

`Year`, `Driver`, `Stint`, `SourceFile`, target-lap identifiers and target-lap data are deliberately **excluded** from model inputs. Categorical unknowns are ignored on held-out circuits. All imputers/scalers/encoders are fit only on the training split.

In [ ]:
BASE_NUM = ["PreviousLapTime_s", "TyreAge_laps", "TrackTemp_C", "PreviousLapNumber"]
BASE_CAT = ["TyreCompound", "Event"]
TELEM_NUM = ["AvgSpeed_kmh", "MaxSpeed_kmh", "FullThrottle_pct", "Braking_pct"]
FEATURE_SETS = {"B_context": (BASE_NUM, BASE_CAT), "C_telemetry": (BASE_NUM + TELEM_NUM, BASE_CAT)}

def make_pipeline(model, numerical: list[str], categorical: list[str]) -> Pipeline:
    """Return preprocessing and regression pipeline fit on training rows only."""
    num = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                    ("scale", StandardScaler())])
    cat = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([("numeric", num, numerical), ("categorical", cat, categorical)])
    return Pipeline([("preprocessor", prep), ("regressor", model)])

def candidate_models():
    """Compact validation grid; bounded for a student laptop CPU."""
    return {
        "LinearRegression": [("default", LinearRegression())],
        "DecisionTree": [(f"depth={d},leaf={leaf}", DecisionTreeRegressor(
            max_depth=d, min_samples_leaf=leaf, random_state=SEED))
            for d in [5, 10] for leaf in [5, 15]],
        "RandomForest": [(f"trees={trees},leaf={leaf}", RandomForestRegressor(
            n_estimators=trees, min_samples_leaf=leaf, n_jobs=-1, random_state=SEED))
            for trees in [75, MAX_RF_TREES] for leaf in [5, 15]]
    }

if REQUIRE_TELEMETRY and clean[TELEM_NUM].isna().any(axis=1).any():
    raise ValueError("Required telemetry missing. Fix exporter or set REQUIRE_TELEMETRY=False.")
if clean[TELEM_NUM].notna().sum().sum() == 0:
    print("WARNING: No previous-lap telemetry detected. C ablation is mathematically uninformative; do not claim telemetry benefits.")

## 10 | Train candidates and tune on validation **only**

In [ ]:
def evaluate_split(df: pd.DataFrame, predicted_delta: np.ndarray) -> dict[str,float]:
    """Evaluate reconstructed next-lap seconds and delta seconds."""
    actual_next = df["NextLapTime_s"].to_numpy(dtype=float)
    previous = df["PreviousLapTime_s"].to_numpy(dtype=float)
    predicted_next = previous + np.asarray(predicted_delta, dtype=float)
    return {"MAE_s": mean_absolute_error(actual_next, predicted_next),
            "RMSE_s": np.sqrt(mean_squared_error(actual_next, predicted_next)),
            "R2_next_lap": r2_score(actual_next, predicted_next),
            "MAE_delta_s": mean_absolute_error(df["DeltaLapTime_s"], predicted_delta)}

rows = []
valid_pred = {}
fitted = {}
base_delta = np.zeros(len(val_df))
rows.append({"features": "A_persistence", "model": "Persistence", "params": "none",
             "fit_seconds": 0., "predict_seconds": 0., **evaluate_split(val_df, base_delta)})
valid_pred[("A_persistence", "Persistence", "none")] = base_delta
for feature_name, (numeric, categorical) in FEATURE_SETS.items():
    for model_name, variants in candidate_models().items():
        for params, regressor in variants:
            pipeline = make_pipeline(regressor, numeric, categorical)
            start = time.perf_counter()
            pipeline.fit(train_df[numeric + categorical], train_df["DeltaLapTime_s"])
            fit_duration = time.perf_counter() - start
            start = time.perf_counter()
            delta_pred = pipeline.predict(val_df[numeric + categorical])
            pred_duration = time.perf_counter() - start
            key = (feature_name, model_name, params)
            fitted[key] = pipeline
            valid_pred[key] = delta_pred
            rows.append({"features": feature_name, "model": model_name,
                         "params": params, "fit_seconds": fit_duration,
                         "predict_seconds": pred_duration,
                         **evaluate_split(val_df, delta_pred)})
comparison = pd.DataFrame(rows).sort_values(["MAE_s", "RMSE_s"]).reset_index(drop=True)
comparison.to_csv(METRIC_DIR / "validation_grid.csv", index=False)
display(comparison.round(4))

## 11 | Model selection and telemetry ablation

Choose the **lowest validation MAE**, including persistence as an eligible winner. Compare the best B and C configurations on the same validation rows. A telemetry improvement is an *association*, not proof of physical causation. Do not select models using 2025 results.

In [ ]:
best = comparison.iloc[0]
best_key = (best["features"], best["model"], best["params"])
print("Selected from validation:", best_key, "| MAE (s):", round(best["MAE_s"], 4))
per_family = (comparison[comparison["features"].isin(FEATURE_SETS)]
              .sort_values("MAE_s").groupby("features", as_index=False).first())
display(per_family[["features", "model", "params", "MAE_s", "RMSE_s", "R2_next_lap"]])
if len(per_family) == 2:
    b_err = per_family.set_index("features").loc["B_context", "MAE_s"]
    c_err = per_family.set_index("features").loc["C_telemetry", "MAE_s"]
    print(f"Validation telemetry ablation (B MAE - C MAE): {b_err-c_err:+.4f} s (positive favors telemetry).")

fig, ax = plt.subplots(figsize=(10, 4))
comparison.groupby(["features", "model"])["MAE_s"].min().sort_values().plot.bar(ax=ax)
ax.set(xlabel="Features / algorithm", ylabel="Validation MAE (s)", title="Model comparison: held-out validation season")
plt.xticks(rotation=30, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "validation_mae_comparison.png", dpi=160)
plt.show()

## 12 | Validation residual diagnostics and engineering breakdown

In [ ]:
def prediction_frame(frame: pd.DataFrame, pred_delta: np.ndarray) -> pd.DataFrame:
    """Create auditable per-lap records from a split and delta predictions."""
    keep = ["Year", "Event", "Session", "Driver", "Stint", "PreviousLapNumber",
            "TargetLapNumber", "TyreCompound", "TyreAge_laps", "PreviousLapTime_s", "NextLapTime_s"]
    out = frame[keep].copy()
    out["PredictedDeltaLapTime_s"] = pred_delta
    out["PredictedNextLapTime_s"] = out["PreviousLapTime_s"] + pred_delta
    out["Residual_s"] = out["NextLapTime_s"] - out["PredictedNextLapTime_s"]
    out["AbsoluteError_s"] = out["Residual_s"].abs()
    return out

val_predictions = prediction_frame(val_df, valid_pred[best_key])
val_predictions.to_csv(PRED_DIR / "validation_predictions.csv", index=False)
display(val_predictions.head())
display(val_predictions.groupby(["Year", "Event"]).agg(
    n=("AbsoluteError_s", "size"), MAE_s=("AbsoluteError_s", "mean"),
    Bias_s=("Residual_s", "mean")).round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(val_predictions["NextLapTime_s"], val_predictions["PredictedNextLapTime_s"], s=7, alpha=.4)
limits = [min(val_predictions["NextLapTime_s"].min(), val_predictions["PredictedNextLapTime_s"].min()),
          max(val_predictions["NextLapTime_s"].max(), val_predictions["PredictedNextLapTime_s"].max())]
ax[0].plot(limits, limits, "k--", lw=1)
ax[0].set(xlabel="Measured next-lap time (s)", ylabel="Predicted (s)", title="Validation: measured vs predicted")
ax[1].scatter(val_predictions["TyreAge_laps"], val_predictions["Residual_s"], s=7, alpha=.4)
ax[1].axhline(0, color="k", ls="--", lw=1)
ax[1].set(xlabel="Tyre age at end of lap n (laps)", ylabel="Measured − predicted (s)", title="Validation residuals vs tyre age")
fig.tight_layout(); fig.savefig(FIG_DIR / "validation_diagnostics.png", dpi=160); plt.show()

## 13 | Runtime, sensitivity and robustness

In [ ]:
# Compare fitted cost and validation error across tree complexity settings.
complexity = comparison.loc[comparison["model"].isin(["DecisionTree", "RandomForest"])].copy()
display(complexity[["features", "model", "params", "MAE_s", "fit_seconds", "predict_seconds"]].sort_values("MAE_s").round(4))
# Percent improvement relative to persistence: a negative result is valid and must be reported.
persist_mae = float(comparison.loc[comparison["model"] == "Persistence", "MAE_s"].iloc[0])
relative_gain = 100. * (persist_mae - float(best["MAE_s"])) / persist_mae if persist_mae else float("nan")
print(f"MAE improvement over previous-lap persistence (validation): {relative_gain:+.2f}%")

## 14 | Reserve final 2025 holdout until model selection is frozen

Run only after validation-based decisions are final. This evaluates **exactly once** against held-out events from 2025, if your local dataset contains them. This notebook retains the model fit on 2022–23 only (no inadvertent test fitting). In a later, explicitly documented production step you may refit on train+validation, but do not conflate that with these holdout figures.

In [ ]:
if RUN_FINAL_TEST:
    if test_df.empty:
        raise ValueError("RUN_FINAL_TEST=True but no holdout events were found. Acquire 2025 clean-lap exports.")
    if best_key[1] == "Persistence":
        test_delta = np.zeros(len(test_df))
    else:
        ncols, ccols = FEATURE_SETS[best_key[0]]
        test_delta = fitted[best_key].predict(test_df[ncols + ccols])
    test_score = evaluate_split(test_df, test_delta)
    test_predictions = prediction_frame(test_df, test_delta)
    test_predictions.to_csv(PRED_DIR / "FINAL_holdout_2025_predictions.csv", index=False)
    (METRIC_DIR / "FINAL_holdout_2025_metrics.json").write_text(json.dumps(
        {"model": list(best_key), "scores": test_score, "rows": len(test_df)}, indent=2))
    print("FINAL independent 2025 holdout scores:", test_score)
    display(test_predictions.groupby(["Year", "Event"]).agg(
        n=("AbsoluteError_s", "size"), MAE_s=("AbsoluteError_s", "mean")).round(3))
else:
    print("Final holdout intentionally locked. Decide features/parameters from validation only.")

## 15 | Save selected model, versions and reproducibility manifest

In [ ]:
model_manifest = {
    "project": "MMA3001 F1 conditional next-clean-lap prediction",
    "selected_from": "validation only", "selected_key": list(best_key),
    "seed": SEED, "train_years": list(TRAIN_YEARS), "validation_years": list(VAL_YEARS),
    "test_years": list(TEST_YEARS), "final_test_executed": RUN_FINAL_TEST,
    "python": sys.version.split()[0], "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "validation_mae_s": float(best["MAE_s"]),
    "feature_availability_warning": bool(clean[TELEM_NUM].notna().sum().sum() == 0),
}
(METRIC_DIR / "model_manifest.json").write_text(json.dumps(model_manifest, indent=2))
if best_key[1] == "Persistence":
    print("Persistence wins: no fitted estimator needed; save manifest only.")
else:
    output_model = MODEL_DIR / "selected_delta_model.joblib"
    joblib.dump(fitted[best_key], output_model)
    print("Saved fitted model:", output_model)
print("Results:", RESULTS_DIR.resolve())

## 16 | Engineer's conclusion and limitations — fill in after execution

1. **Baseline:** State persistence MAE in seconds.
2. **Selected method:** Identify model, configuration and validation MAE/RMSE/R²; compare compute time.
3. **Telemetry contribution:** Report B vs C MAE on identical observations; **do not claim a telemetry benefit if the exporter contains no telemetry**.
4. **Generalisability:** If the holdout was enabled, show independent 2025 event-level errors, not only pooled accuracy.
5. **Physical meaning:** Explain tyres, temperature, traffic, fuel mass and circuit effects, acknowledging confounding and unobserved conditions.
6. **Research boundaries:** Retrospectively clean target laps, no guaranteed forecasting of pits/neutralisation; unknown circuits and sparse operating regions warrant caution.
7. **Data provenance:** Source is FastF1 historical timing/telemetry; audit export logic separately for timestamp and track-status leakage.
8. **AI use:** Document ChatGPT-generated code structure, student validation, corrections and decisions in your report. Your MMA3001 brief requires explicit AI-use reflection.

### Further scope (not included in assessed P0)
Tyre-strategy optimisation needs a **separate future-pace model** and explicit pit-loss assumptions. Do **not** feed unknown future telemetry through this single-step model recursively. The development plan treats race strategy as a conditional extension once core validation passes.

### Reproducibility checklist
- [ ] Existing exporter checked for weather time and previous-lap-only telemetry
- [ ] At least two distinct real races and chronology established
- [ ] All invalid pairs and missing values quantified
- [ ] Model selection conducted without reading test errors
- [ ] Test executed once and results interpreted per event
- [ ] CSV and model outputs not published contrary to data rights
- [ ] GitHub README, source docstrings, tests/HTML docs and AI reflection finalized